# Things about classes

## Singleton
singleton pattern is a software design pattern that restricts the instantiation of a class to one "single" instance

In [1]:
class Singleton:
    _instance = None
    
    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            cls._instance = super().__new__(cls, *args, **kwargs)
            
        return cls._instance

print('Singleton Examle')
x = Singleton()
y = Singleton()
print(f'{x is y=}')

Singleton Examle
x is y=True


### Using metaclass

In [2]:
class Singleton(type):
    _instance = None

    def __call__(cls, *args, **kwds):
        if not cls._instance:
            cls._instance = super().__call__(*args, **kwds)
        return cls._instance

class A(metaclass=Singleton):
    pass

print('Singleton Examle')
x = A()
y = A()
print(f'{x is y=}')

Singleton Examle
x is y=True


## Class factory using metaclasses
If you want to register every animal class you create, you’d need to call register for each one this class.

In [3]:
class AnimalFactory:
    animals = {}

    @classmethod
    def from_name(cls, name):
        animal_class = cls.animals.get(name.lower())
        if not animal_class:
            raise ValueError("Animal not found")

        return animal_class()

    @classmethod
    def register(cls, name, animal_class):
        cls.animals[name] = animal_class

One way of doing this, and keep your code clean, is to use a metaclass that will register the class into AnimalFactory once it’s created (the class, not the object).

In [4]:
class AnimalMeta(type):
    def __new__(cls, name, bases, namespace):
        new_cls = super().__new__(cls, name, bases, namespace)
        AnimalFactory.register(name.lower(), new_cls)
        return new_cls

Now you assign this metaclass to Animal class, that way all classes that derive from Animal will use this metaclass and be registered in AnimalFactory class.

In [5]:
class Animal(metaclass=AnimalMeta):
    name = "animal"

    def sound(self):
        print(f"Hey, I'm {self.name}!")


class Dog(Animal):
    name = "Rex"


class Cat(Animal):
    name = "Kitty"


cat = AnimalFactory.from_name("cat")
cat.sound()
dog = AnimalFactory.from_name("dog")
dog.sound()

Hey, I'm Kitty!
Hey, I'm Rex!


## Extending dict with classes

### Custom dict like class

In [6]:
class ExtendedDict(dict):
    def apply(self, action):
        for key, value in self.items():
            self[key] = action(value)

    def remove(self, key):
        del self[key]

    def is_empty(self):
        return len(self) == 0


numbers = ExtendedDict({"one": 1, "two": 2, "three": 3})
print("\t\t raw dict:", numbers)

numbers.apply(lambda x: x**2)
print("apply x**2 to every value:", numbers)

numbers.remove("two")
print("\t     remove a key:", numbers)

numbers.is_empty()

		 raw dict: {'one': 1, 'two': 2, 'three': 3}
apply x**2 to every value: {'one': 1, 'two': 4, 'three': 9}
	     remove a key: {'one': 1, 'three': 9}


False

### make dict keys accessible as attributes

In [7]:
raw_dict = {"one": 1, "two": 2, "three": 3}
raw_dict["one"], raw_dict.get("one") #, raw_dict.one <- this raises an error

(1, 1)

In [8]:
class AttrDict(dict):
    def __init__(self, **kwargs):
        dict.__init__(self, kwargs)
        self.__dict__ = self

attr_dict = AttrDict(**{"one": 1, "two": 2, "three": 3})

attr_dict["one"], attr_dict.get("one"), attr_dict.one

(1, 1, 1)

### using [UserDict](https://docs.python.org/3/library/collections.html#collections.UserDict)

- UserDict is a convenient wrapper around a regular dict object, specially designed for subclassing purposes
- Inheriting from UserDict may imply a performance cost because this class is written in pure Python

In [9]:
from collections import UserDict


class UpperCaseDict(UserDict):
    def __setitem__(self, key, value):
        key = key.upper()
        super().__setitem__(key, value)

numbers = UpperCaseDict({"one": 1, "two": 2})

numbers["three"] = 3
numbers.update({"four": 4})
numbers.setdefault("five", 5)

print(numbers)

{'ONE': 1, 'TWO': 2, 'THREE': 3, 'FOUR': 4, 'FIVE': 5}
